# Finite self-training–oracle discrepancy bound

This notebook explains and exercises the additive diagnostic implemented in `oracle_discrepancy_diagnostics.py`. It compares a self-training trajectory with a fully coupled selected-label oracle trajectory and evaluates every computable finite-dimensional inequality in the paper's discrepancy argument.

%load_ext autoreload
%autoreload 2

## 1. Coupling and scope

The two learners share exactly the same design, Gaussian noise, true labels, supervision mask, initialization, initial pseudo-labels, and hyperparameters. After initialization, each learner evolves from its own preactivations, confidence selections, empirical selection rate, residuals, and parameter iterates. The oracle is therefore a complete counterfactual trajectory—not a one-step label replacement at the self-training state.

The implementation validates balanced labels, a fixed pseudo-label weight, symmetric hard selection, logistic loss, and ridge regularization. This notebook uses fixed zero bias for the paper's original alignment-only comparison. Learned-bias experiments are also supported: when the bias pseudo-label weight differs from the weight pseudo-label weight, the diagnostic propagates the bias residual separately and bounds population-error differences through both normalized alignment and normalized bias.

In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import display
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import logit

from notebooks.experiment_helpers import make_algorithm_config, run_experiment
from notebooks.oracle_discrepancy_diagnostics import (
    format_oracle_discrepancy_report,
    plot_oracle_discrepancy_diagnostic,
    run_finite_oracle_discrepancy_diagnostic,
)

## 2. Reduced-cost canonical configuration

The parameters below are the balanced, fixed-zero-bias specialization of the confidence-threshold experiments: $\delta=2$, $\rho=0.1$, $\eta=0.1$, $\lambda=0.1$, $\pi=5$, and $\kappa=\operatorname{logit}(0.8)$. The finite dimension and horizon are reduced so the notebook executes quickly.

`RUN_STATE_EVOLUTION=True` additionally runs the paired particle processes needed to inspect the fresh-innovation selection-rate floor. It does not change the finite trajectory or the finite empirical bound. The three expensive sweep flags remain opt-in.

In [2]:
SEED = 3100
T = 20
D = 100
DELTA = 2.0
RHO = 0.1
SIGMA = 1.0
PI = 5.0
KAPPA = float(logit(0.8))
RUN_STATE_EVOLUTION = True
RUN_REPEATED_TRIALS = False
RUN_DIMENSION_SWEEP = False
RUN_PARAMETER_SWEEP = False

algo_cfg = make_algorithm_config(
    T=T, eta=0.1, penalty=0.1, pi=PI, kappa=KAPPA,
    include_bias=False, initial_bias=0.0,
)

run_kwargs = dict(
    name="finite ST/oracle discrepancy",
    d=D, delta=DELTA, n_test=500, label_prior=0.5, rho=RHO,
    sigma=SIGMA, signal_std=1.0, algo_cfg=algo_cfg, seed=SEED,
)
if RUN_STATE_EVOLUTION:
    run_kwargs.update(K_w=256, K_g=320)
else:
    run_kwargs.update(run_state_evolution=False)

run = run_experiment(**run_kwargs)

## 3. Construct the paired trajectories and all diagnostics

The call below leaves `run.finite` unchanged. It creates the selected-label oracle and supervised baseline from the same realized environment and initialization, reconstructs $U$ from the stored design, computes the exact spectral norm, and evaluates the complete diagnostic chain.

In [3]:
diagnostic = run_finite_oracle_discrepancy_diagnostic(
    run,
    grid_size=121,
    compute_state_evolution_lower_bound=RUN_STATE_EVOLUTION,
)
print(format_oracle_discrepancy_report(diagnostic))

Finite ST/oracle discrepancy diagnostic
all intended finite inequalities satisfied: True
actual oracle gap: 0.140793
smallest oracle-gap upper bound: 0.30988
minimizing comparison time: 0
bound below trivial upper bound 1: True
oracle gain: 0.144068
bound certifies positive self-training gain: False
largest measured slack stage: alignment from recursion
upper-bound / positive oracle-gap ratio: 2.20096


## 4. Result organization

The returned object separates state-time quantities (times $0,\ldots,T$), update-time quantities (times $0,\ldots,T-1$), scalar constants, non-vacuity summaries, and the full $h$-grid optimization records.

- `diagnostic.state`: weights, biases, $Z_t$, alignments, population errors, recursive bounds, and oracle regrets.
- `diagnostic.update`: $E_r^t,E_g^t$, forcing/propagation, selection rates, $\Xi_t$, and all elementary inequality right-hand sides.
- `diagnostic.constants`: exact, empirical, and state-evolution constants.
- `diagnostic.summary`: gains, gap, smallest bound, comparison time, Boolean checks, and slack attribution.
- `diagnostic.hard_selector_bounds[t]`: the complete positive $h$ grid and objective values at update $t$.

In [4]:
constant_rows = {
    "realized rho": diagnostic.constants["rho_empirical"],
    "||mu||_d": diagnostic.constants["mu_norm_d"],
    "||U||_op": diagnostic.constants["U_operator_norm_exact"],
    "a_d": diagnostic.constants["a_d"],
    "b_d": diagnostic.constants["b_d"],
    "c_w": diagnostic.constants["c_w"],
    "Lip(ell') (exact)": diagnostic.constants["loss_gradient_lipschitz_exact"],
    "||ell'||_infinity (exact)": diagnostic.constants["loss_gradient_bound_exact"],
    "Lip(j') (exact)": diagnostic.constants["penalty_gradient_lipschitz_exact"],
    "empirical omega floor": diagnostic.constants["omega_lower_empirical_full_horizon"],
    "SE v_min": diagnostic.constants["v_min_state_evolution_full_horizon"],
    "theoretical omega floor": diagnostic.constants["omega_lower_theoretical_full_horizon"],
    "log theoretical omega floor": diagnostic.constants["log_omega_lower_theoretical_full_horizon"],
}
display(pd.Series(constant_rows, name="value").to_frame())

,value
realized rho,0.125000
||mu||_d,1.074115
||U||_op,23.622612
a_d,2.744486
b_d,2.744486
c_w,1.010000
Lip(ell') (exact),0.250000
||ell'||_infinity (exact),1.000000
Lip(j') (exact),1.000000
empirical omega floor,0.102857


## 5. Exact forcing decomposition

At the self-training state, the code recomputes an oracle-target residual while freezing the ST scores, selection mask, and selection rate. Thus

$$g_{\mathrm{ST}}^t-g_{\mathrm{oracle}}^t=e_{\mathrm{PL}}^t+e_{\mathrm{prop}}^t.$$

For the repository's logistic normalization, $\operatorname{Lip}(\ell')=1/4$, $\|\ell'\|_\infty=1$, and the selected-label forcing formula is an exact identity.

In [5]:
identity_table = pd.DataFrame({
    "E_g": diagnostic.update["E_g"],
    "||e_PL||_n": diagnostic.update["forcing_norm"],
    "forcing formula": diagnostic.update["forcing_formula"],
    "||e_prop||_n": diagnostic.update["propagation_norm"],
    "decomposition error": diagnostic.update["decomposition_error"],
    "forcing identity error": diagnostic.update["forcing_identity_error"],
})
identity_table.index.name = "t"
display(identity_table.head(10))
print("maximum decomposition error:", identity_table["decomposition error"].max())
print("maximum forcing identity error:", identity_table["forcing identity error"].max())

,E_g,||e_PL||_n,forcing formula,||e_prop||_n,decomposition error,forcing identity error
t,,,,,,
0,0.938083,0.938083,0.938083,0.000000,0.000000e+00,1.110223e-16
1,0.443542,0.962250,0.962250,0.738017,5.495324e-17,2.220446e-16
2,0.202140,0.657667,0.657667,0.595003,1.922963e-17,0.000000e+00
3,0.150011,0.566558,0.566558,0.515795,3.444376e-17,1.110223e-16
4,0.149592,0.497930,0.497930,0.466806,1.749922e-17,1.110223e-16
5,0.109309,0.455543,0.455543,0.416631,2.090900e-17,0.000000e+00
6,0.125502,0.400000,0.400000,0.361085,2.538156e-17,0.000000e+00
7,0.089779,0.375000,0.375000,0.345884,1.867710e-17,0.000000e+00
8,0.104649,0.371350,0.371350,0.332646,2.271892e-17,5.551115e-17


maximum decomposition error: 5.4953236053932123e-17
maximum forcing identity error: 2.220446049250313e-16


## 6. Elementary finite-dimensional inequalities

The following table compares each left-hand side with its stored right-hand side. The slack ratio is `RHS/LHS`; it is infinite when the left-hand side is exactly zero but the bound is positive. In this fixed-zero-bias experiment, that convention makes the bias ratio infinite even though $E_b^t=0$ identically.

In [6]:
inequality_table = pd.DataFrame({
    "E_r lhs": diagnostic.update["E_r"],
    "E_r rhs": diagnostic.update["forward_rhs"],
    "forward ratio": diagnostic.update["forward_slack_ratio"],
    "E_b next lhs": diagnostic.state["E_b"][1:],
    "E_b next rhs": diagnostic.update["bias_rhs"],
    "bias ratio": diagnostic.update["bias_slack_ratio"],
    "E_w next lhs": diagnostic.state["E_w"][1:],
    "E_w next rhs": diagnostic.update["weight_rhs"],
    "weight ratio": diagnostic.update["weight_slack_ratio"],
})
inequality_table.index.name = "t"
display(inequality_table.head(10))
display(pd.Series(diagnostic.summary["exact_checks"], name="satisfied").to_frame())

,E_r lhs,E_r rhs,forward ratio,E_b next lhs,E_b next rhs,bias ratio,E_w next lhs,E_w next rhs,weight ratio
t,,,,,,,,,
0,0.000000,0.000000,1.000000,0.0,0.938083,inf,0.676266,2.574556,3.807016
1,0.867067,1.856003,2.140552,0.0,0.443542,inf,0.866390,1.900325,2.193384
2,1.128853,2.377794,2.106380,0.0,0.202140,inf,0.981655,1.429823,1.456543
3,1.295403,2.694138,2.079768,0.0,0.150011,inf,1.060578,1.403173,1.323027
4,1.411294,2.910743,2.062464,0.0,0.149592,inf,1.117196,1.481737,1.326300
5,1.489869,3.066128,2.057985,0.0,0.109309,inf,1.159904,1.428366,1.231452
6,1.544595,3.183340,2.060955,0.0,0.125502,inf,1.194324,1.515942,1.269289
7,1.584246,3.277807,2.069001,0.0,0.089779,inf,1.224104,1.452664,1.186716
8,1.617795,3.359536,2.076614,0.0,0.104649,inf,1.256047,1.523553,1.212975


,satisfied
shared_weight_initialization,True
shared_bias_initialization,True
shared_initial_pseudo_labels,True
residual_decomposition,True
logistic_forcing_identity,True
forward_inequality,True
bias_inequality,True
weight_inequality,True
selector_inequality,True
selector_weight_inequality_empirical,True


## 7. Hard-selector bound and $h$ optimization

For every positive grid point, the diagnostic evaluates

$$\Xi_t^{\mathrm{bd}}(h)=B_{t,n}(h)+\frac{(E_r^t)^2}{(1-\rho)h^2}.$$

The stable grid combines a logarithmic score scale, observed distances to the boundaries $\{-\kappa,\kappa\}$ (including points immediately below jumps of $B_{t,n}$), and the theoretical scale $(E_r^t)^{2/3}$. The stored `Xi_bound` is the requested uncapped grid minimum. The propagation modulus may additionally use the exact trivial inequality $\Xi_t\leq1$.

In [7]:
selector_table = pd.DataFrame({
    "Xi actual": diagnostic.update["Xi"],
    "inf_h Xi_bd": diagnostic.update["Xi_bound"],
    "effective bound": diagnostic.update["Xi_effective_bound"],
    "minimizing h": diagnostic.update["h_min"],
    "boundary mass": diagnostic.update["boundary_mass_at_h_min"],
    "bound / actual": diagnostic.update["Xi_bound_ratio"],
})
selector_table.index.name = "t"
display(selector_table)

chosen_t = int(np.nanargmax(diagnostic.update["Xi"]))
chosen_bound = diagnostic.hard_selector_bounds[chosen_t]
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(chosen_bound.grid, chosen_bound.values, label=r"$\Xi_t^{\rm bd}(h)$")
ax.axhline(chosen_bound.actual_discrepancy, color="black", linestyle="--", label=r"actual $\Xi_t$")
ax.axvline(chosen_bound.minimizing_h, color="red", linestyle=":", label=r"minimizing $h_t$")
ax.set(xscale="log", yscale="log", xlabel=r"$h$", ylabel="selector discrepancy", title=rf"Hard-selector optimization at $t={chosen_t}$")
ax.grid(True, which="both", linestyle=":", alpha=0.6)
ax.legend();

,Xi actual,inf_h Xi_bd,effective bound,minimizing h,boundary mass,bound / actual
t,,,,,,
0,0.000000,0.000000,0.0,0.000000,0.0,1.000000
1,0.200000,1.000045,1.0,138.629436,1.0,5.000224
2,0.320000,1.000041,1.0,188.790325,1.0,3.125128
3,0.377143,1.000036,1.0,229.825124,1.0,2.651611
4,0.417143,1.000034,1.0,258.038521,1.0,2.397342
5,0.394286,1.000033,1.0,278.108450,1.0,2.536315
6,0.371429,1.000031,1.0,295.480808,1.0,2.692392
7,0.360000,1.000028,1.0,317.586099,1.0,2.777857
8,0.348571,1.000028,1.0,326.567163,1.0,2.868933


## 8. Propagation and closed recursion

The empirical recursive diagnostic uses the common realized floor
$\underline\omega_{\mathrm{emp}}=\min_t\min\{\omega_{\mathrm{ST}}^t,\omega_{\mathrm{oracle}}^t\}$. For a candidate radius $x$, it recomputes the explicit selector bound with $x$ replacing $E_r^t$, converts it to a normalized-selector bound, and inserts that result into the logistic propagation inequality. This defines the fully computable post-hoc modulus $\widehat\Psi_t(x)$.

This is a valid statement about the realized finite trajectory, but the observed floor is not a deterministic a priori lower bound. The state-evolution floor is shown separately and is an asymptotic high-probability device.

In [8]:
fig, axes = plot_oracle_discrepancy_diagnostic(
    diagnostic,
    title="Finite self-training/oracle discrepancy diagnostic",
    show=False,
)
plt.show()

/var/folders/1v/q141c9gj6fxf_hkyjl6w9y8c0000gn/T/ipykernel_19628/342314659.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Non-vacuity and source of looseness

The final table distinguishes three questions: whether the bound is finite, whether it is below the trivial classification-error upper bound $1$, and whether it is strong enough to certify a positive self-training gain by being smaller than the oracle gain. When the actual oracle gap is nonpositive, the bound/gap ratio is intentionally omitted.

In [9]:
summary_rows = {
    "self-training gain": diagnostic.summary["self_training_gain"],
    "oracle gain": diagnostic.summary["oracle_gain"],
    "actual oracle gap": diagnostic.summary["oracle_gap"],
    "alignment gap": diagnostic.summary["alignment_gap"],
    "smallest oracle-gap upper bound": diagnostic.summary["oracle_gap_upper_bound"],
    "bound / positive gap": diagnostic.summary["oracle_gap_upper_bound_ratio"],
    "minimizing comparison time": diagnostic.summary["best_comparison_time"],
    "bound below 1": diagnostic.summary["bound_smaller_than_trivial_one"],
    "certifies positive ST gain": diagnostic.summary["bound_certifies_self_training_gain"],
    "largest measured slack stage": diagnostic.summary["largest_loss_stage"],
}
display(pd.Series(summary_rows, name="value").to_frame())
display(pd.Series(diagnostic.summary["stage_max_slack_ratio"], name="maximum RHS/LHS").to_frame())

,value
self-training gain,0.003275
oracle gain,0.144068
actual oracle gap,0.140793
alignment gap,0.409515
smallest oracle-gap upper bound,0.30988
bound / positive gap,2.200963
minimizing comparison time,0
bound below 1,True
certifies positive ST gain,False
largest measured slack stage,alignment from recursion


,maximum RHS/LHS
forward,2.140552e+00
bias,inf
weight,3.807016e+00
hard selector,5.000224e+00
propagation,1.851932e+02
residual,9.558976e+02
closed recursion,3.698461e+24
alignment from E_w,9.674118e+00
alignment from recursion,3.577935e+25


## 10. Interpretation of this run

For this reduced canonical realization, all exact and finite-dimensional inequalities are satisfied. The final minimized oracle-gap bound is below $1$, so it is numerically nontrivial under that criterion, but it is larger than the oracle gain and therefore does not certify that self-training improves over supervised learning.

The fully propagated recursion grows rapidly after the first update. The principal numerical mechanism is repeated amplification through the normalized-selector and propagation bounds; the alignment conversion then compounds this loss. The minimizing comparison time is consequently $t_0=0$, where the recursive discrepancy is zero and the bound is driven entirely by the oracle's initial alignment regret.

The fresh-innovation state-evolution floor can be far smaller than the observed empirical floor. If its float64 value underflows, inspect `log_omega_lower_theoretical_full_horizon`; the corresponding ordinary-scale propagation bound is then reported as infinity rather than replaced by a fabricated finite constant.

## 11. Optional sweeps

The command-line runner provides the same canonical diagnostic and keeps expensive extensions explicit:

```bash
MPLBACKEND=Agg MPLCONFIGDIR=/private/tmp/oracle-mpl PYTHONPATH=. \
python3 scripts/oracle_discrepancy_diagnostic.py --output-dir /private/tmp/oracle-bound
```

Add any of the following flags only when desired:

- `--repeated-trials`: five paired finite trials;
- `--dimension-sweep`: $d\in\{50,100,200,400\}$, reporting the gap, bound, and ratio;
- `--parameter-sweep`: a small $3\times3$ grid in $(\pi,\kappa)$;
- `--without-state-evolution`: skip the particle innovation-floor diagnostic.

The CLI writes a trajectory PDF and JSON summary, plus dimension- and parameter-sweep PDFs when those flags are enabled.